In [1]:
import os
import pandas as pd
import numpy as np
import scipy.ndimage as ndimage
from tqdm import tqdm

def rle_decode(mask_rle, shape=(240, 240, 155)):
    if pd.isna(mask_rle) or mask_rle == '': return np.zeros(shape, dtype=np.uint8)
    s = mask_rle.split()
    starts, lengths = [np.asarray(x, dtype=int) for x in (s[0:][::2], s[1:][::2])]
    starts -= 1
    ends = starts + lengths
    img = np.zeros(shape[0]*shape[1]*shape[2], dtype=np.uint8)
    for lo, hi in zip(starts, ends): img[lo:hi] = 1
    return img.reshape(shape)

def rle_encode(mask):
    pixels = mask.flatten()
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return ' '.join(str(x) for x in runs)

SUBMISSIONS_PATH = "/kaggle/input/my-submissions"
WEIGHTS = {
    'submission (2).csv': 0.70,
    'submission_TOP_RANK.csv': 0.15,
    'submission_MAGIC_FIX.csv': 0.15
}

dfs = {f: pd.read_csv(os.path.join(SUBMISSIONS_PATH, f)) for f in WEIGHTS.keys()}
base_df = dfs['submission (2).csv']
final_rows = []

for i in tqdm(range(len(base_df))):
    id_val = base_df.iloc[i]['id']
    weighted_mask = np.zeros((240, 240, 155), dtype=float)
    
    for f_name, weight in WEIGHTS.items():
        weighted_mask += rle_decode(dfs[f_name].iloc[i]['rle']) * weight
    
    threshold = 0.45 if id_val.endswith('_4') else 0.55
    final_mask = (weighted_mask >= threshold).astype(np.uint8)
    final_mask = ndimage.binary_fill_holes(final_mask).astype(np.uint8)
    
    final_rows.append([id_val, rle_encode(final_mask)])

submission_df = pd.DataFrame(final_rows, columns=['id', 'rle'])
submission_df.to_csv('submission_FINAL_CLEAN.csv', index=False)

100%|██████████| 1002/1002 [07:07<00:00,  2.34it/s]
